# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You will learn how to load the Croissant schema, review the dataset structure, extract records by referencing entities through their `@id`, and perform exploratory data analysis.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed and import dependencies
!pip install --quiet mlcroissant
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data Loading

Load the metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (as an object, not a dict!)
meta_obj = dataset.metadata
print(f"{meta_obj.name}: {meta_obj.description}")

## 2. Data Overview

Review available record sets, their `@id`s, the fields within each record set, and which columns (data fields) are available for each. All entities are referenced by their unique `@id` field as per Croissant/FAIR^2 best practice.

Here, we list all record sets, and for each, print its fields and column `@id`s.

In [ ]:
# List all record sets and their info
record_sets = list(dataset.record_sets())  # Each is an mlcroissant.RecordSet
if not record_sets:
    print("No record sets found in the dataset.")
else:
    for rs in record_sets:
        print(f"\nRecordSet @id: {rs.id}\n  Name: {rs.name}")
        print(f"  Description: {rs.description}")
        # List fields and their `@id`
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field @id: {field.id} | Name: {field.name} | DataType: {field.data_type}")
        # List columns (i.e., fields that correspond to data columns)
        print("  Columns:")
        for col in rs.columns:
            print(f"    - Column @id: {col.id} | Name: {col.name}")

## 3. Data Extraction

We will load records for each available record set, referencing them by their `@id`. This demonstrates how to access each dataset table with `mlcroissant`, with each field also referenced by its `@id` in the resulting DataFrame.

**Note:** Adjust the list of record set `@id`s to include only the record sets you wish to extract data from.

In [ ]:
# Refresh record could be required: get up-to-date RecordSet IDs
record_set_ids = [rs.id for rs in dataset.record_sets()]
if not record_set_ids:
    print("No record sets available in the dataset.")
else:
    # For demonstration, extract all, but you can select subset if you want
    dataframes = {}
    for rs_id in record_set_ids:
        print(f"\nLoading records from RecordSet @id: {rs_id}")
        records_iterator = dataset.records(record_set=rs_id)
        records_list = list(records_iterator)
        if records_list:
            df = pd.DataFrame(records_list)
            print(f"  Fields (columns): {df.columns.tolist()}")
            print(f"  Number of records: {len(df)}")
            print(df.head())
            dataframes[rs_id] = df
        else:
            print("  No records found for this record set.")
    # We'll proceed with the first available record set for EDA
    if dataframes:
        example_rs_id = list(dataframes.keys())[0]
        print(f"\nWill use RecordSet @id '{example_rs_id}' for demonstration below.")
    else:
        example_rs_id = None

## 4. Exploratory Data Analysis (EDA)

We'll perform EDA on one of the available record sets (the first one with data). Operations include:
- Filtering records by a numeric field (referenced by its `@id`)
- Normalizing a numeric field
- Grouping by a key (usually a categorical) field, also by its `@id`

**Remember:** Replace the example field IDs below with specific `@id`s from your overview in Section 2 if needed. This cell will gracefully skip when not enough numeric data is present.

In [ ]:
if example_rs_id is None:
    print("No extracted data available for EDA.")
else:
    df = dataframes[example_rs_id]
    # Attempt to find the first numeric field (`int` or `float`) by inspecting first row
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric fields found in the selected record set for EDA.")
    else:
        print(f"Using numeric field (by @id): {numeric_field_id}\n")
        # Filter: keep only rows above threshold
        threshold = df[numeric_field_id].dropna().mean()  # use mean as an example
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group by another field if available (prefer string/object type fields)
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field}':")
            print(grouped_df.head())
        else:
            print("No categorical group field found for grouping.")

## 5. Visualization

Let us visualize the distribution of the selected numeric field, and its relationship to a (categorical) group field, if found. Again, all fields are referenced by their `@id`.

In [ ]:
if example_rs_id is None or numeric_field_id is None:
    print("No data available for visualization.")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    # Histogram of the numeric variable
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True, ax=ax[0])
    ax[0].set_title(f"Distribution of {numeric_field_id}")
    ax[0].set_xlabel(numeric_field_id)
    # Violin/box plot grouped by group_field, if available
    if group_field:
        # Use only up to 10 categories for legibility
        top_groups = df[group_field].value_counts().index[:10]
        plot_data = df[df[group_field].isin(top_groups)]
        sns.boxplot(data=plot_data, x=group_field, y=numeric_field_id, ax=ax[1])
        ax[1].set_title(f"{numeric_field_id} by {group_field}")
        ax[1].set_xlabel(group_field)
        ax[1].set_ylabel(numeric_field_id)
        plt.setp(ax[1].xaxis.get_majorticklabels(), rotation=45, ha="right")
    else:
        ax[1].text(0.3, 0.5, "No group field", fontsize=12)
        ax[1].set_axis_off()
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to use the `mlcroissant` library to load and explore a dataset described by the Croissant schema. By referencing all dataset entities through their `@id`, we ensure robust and transparent data handling. 

Key steps included:
- Loading metadata and record sets using only their Croissant `@id`
- Exploring fields and columns available for analysis
- Loading record data directly to pandas DataFrames for further manipulation
- Performing EDA and plotting, referencing every field by its unique identifier

**Next steps:** You can extend this notebook by using more domain knowledge to select and analyze pertinent fields, perform statistical tests, or export derived tables. Refer to the Croissant and mlcroissant documentation for more advanced workflows.

_Notebook generated following the mlcroissant and FAIR^2 best practice workflow._